# Video Compression

the behav_analysis_PC envoironment was created to separate video compression libraries from arduino stuff. this notebook should be run in that env.

In [47]:
#file management
import os 
import re
import glob
from datetime import datetime
#data manipulation
import pandas as pd
import numpy as np
#vision/camera tools
import cv2
import ffmpeg

In [31]:
def check_video_format(video_path):
    """
    Check the codec/compression format of a video file.
    
    Parameters
    ----------
    video_path : str
        Path to the video file
        
    Returns
    -------
    dict
        Contains video properties including codec, fps, frame size, etc.
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        raise ValueError("Error: Could not open video file")
    
    try:
        # Get the FourCC code (codec identifier)
        fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
        
        # Convert FourCC to readable format
        codec = "".join([chr((fourcc >> 8 * i) & 0xFF) for i in range(4)])
        
        # Get other video properties
        info = {
            'codec': codec,
            'fps': cap.get(cv2.CAP_PROP_FPS),
            'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            'frame_width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            'frame_height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        }
        
        return info
        
    finally:
        # Always release the video capture object
        cap.release()

In [32]:
file_path = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\390L\20250228152119_390L_Reach2.avi'
check_video_format(file_path)

{'codec': 'MJPG',
 'fps': 60.000240000960005,
 'frame_count': 94765,
 'frame_width': 800,
 'frame_height': 600}

In [33]:
def get_avi_path(directory):
    """
    Get the path to all AVI files in a directory.
    
    Parameters
    ----------
    directory : str
        Path to the directory containing the AVI files
        
    Returns
    -------
    list
        Path to the first AVI file in the directory
    """
    avi_files = glob.glob(os.path.join(directory, "**","*.avi"),recursive=True)
    if len(avi_files) == 0:
        print("No AVI files found in directory")
        return None
    return avi_files

In [34]:
dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig'
avi_files = get_avi_path(dir)
print(avi_files)

['C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250216160904_4934T_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250304151443_390L_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250305175601_771N_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB0\\20250303194343_1FB0_Reach2_day1.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB1\\20250303185557_1FB1_LickReach2_day.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB1\\20250303192438_1FB1_Reach2_day1.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB2\\20250303201725_1FB2_Reach2_day1.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\ca

In [35]:
def get_mp4_path(directory):
    """
    Get the path to all mp4 files in a directory.
    
    Parameters
    ----------
    directory : str
        Path to the directory containing the mp4 files
        
    Returns
    -------
    list
        Path to the first mp4 file in the directory
    """
    mp4_files = glob.glob(os.path.join(directory, "**","*.mp4"),recursive=True)
    if len(mp4_files) == 0:
        print("No mp4 files found in directory")
        return None
    return mp4_files

In [36]:
dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig'
mp4_files = get_mp4_path(dir)
print(mp4_files)

['C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250216160904_4934T_Reach2.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB0\\20250303194343_1FB0_Reach2_day1.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB1\\20250303185557_1FB1_LickReach2_day.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB1\\20250303192438_1FB1_Reach2_day1.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB2\\20250303201725_1FB2_Reach2_day1.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\381B\\20250226161553_381B_Reach2.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\381B\\20250227175929_381B_Reach_day3.mp4', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach

In [37]:
def find_not_converted_files(directory):
    """
    Find all .avi files in a directory that have not been converted to .mp4.
    
    Parameters
    ----------
    directory : str
    """
    avi_paths = get_avi_path(directory)
    mp4_paths = get_mp4_path(directory)
    
    if not avi_paths:
        return []
    
    if not mp4_paths:
        return avi_files
    
    # Convert to numpy arrays for efficient comparison
    avi_files = np.array([os.path.splitext(os.path.basename(f))[0] for f in avi_paths])
    mp4_files = np.array([os.path.splitext(os.path.basename(f))[0] for f in mp4_paths])
    
    mask = ~np.isin(avi_files,mp4_files)
    #if error, check that avi_files mask indexes correctly into avi_paths. it is for now. 
    avi_to_convert = np.array(avi_paths)[mask].tolist()
    return avi_to_convert


In [38]:
dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig'
not_converted = find_not_converted_files(dir)
print(not_converted)
[print(f) for f in not_converted]

['C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250304151443_390L_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\20250305175601_771N_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\1FB2\\20250305204051_1FB2_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\381B\\20250304143931_381B_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\381B\\20250305131847_381B_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\390L\\20250305142725_390L_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav_rig\\402R\\20250304154750_402R_Reach2.avi', 'C:\\Users\\Emma\\MIT Dropbox\\Emma Odom\\Emma\\Reach_Task_Master\\cam_recording\\behav

[None, None, None, None, None, None, None, None, None]

In [25]:
dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig'
not_converted = find_not_converted_files(dir)
print(not_converted)
[print(f) for f in not_converted]

['20250304151443_390L_Reach2', '20250305175601_771N_Reach2', '20250305204051_1FB2_Reach2', '20250304143931_381B_Reach2', '20250305131847_381B_Reach2', '20250305142725_390L_Reach2', '20250304154750_402R_Reach2', '20250305135657_402R_Reach2', '20250304162056_771N_Reach2']
20250304151443_390L_Reach2
20250305175601_771N_Reach2
20250305204051_1FB2_Reach2
20250304143931_381B_Reach2
20250305131847_381B_Reach2
20250305142725_390L_Reach2
20250304154750_402R_Reach2
20250305135657_402R_Reach2
20250304162056_771N_Reach2


[None, None, None, None, None, None, None, None, None]

### heres the stuff!

In [40]:
def convert_for_dlc(input_path, output_path=None):
    """
    Convert video with settings optimized for DeepLabCut analysis.
    Uses high quality compression settings that maintain features 
    important for pose estimation.
    
    Parameters
    ----------
    input_path : str
        Path to input video
    output_path : str, optional
        Path for output file. If None, will replace .avi with .mp4
    """
    if output_path is None:
        output_path = input_path.rsplit('.', 1)[0] + '.mp4'
    
    print(f"Converting: {os.path.basename(input_path)}")
    try:
        # Setup ffmpeg stream with DLC-optimized settings
        stream = (
            ffmpeg
            .input(input_path)
            .output(output_path,
                   vcodec='libx264',    # H.264 codec
                   preset='medium',      # medium compression, not too slow
                   qp=18,              # lossless quality
                   an=None)             # No audio needed for analysis
            .overwrite_output()
        )
        
        # Run the conversion
        stream.run(capture_stdout=True, capture_stderr=True)
        
        # Print file size comparison
        input_size = os.path.getsize(input_path) / (1024 * 1024)  # MB
        output_size = os.path.getsize(output_path) / (1024 * 1024)  # MB
        print(f"Original size: {input_size:.1f} MB")
        print(f"Converted size: {output_size:.1f} MB")
        print(f"Compression ratio: {input_size/output_size:.1f}x")
        
    except ffmpeg.Error as e:
        print(f"Error converting {input_path}:")
        print(e.stderr.decode())

In [41]:
def batch_convert_avis(avi_files):
    """
    Convert a list of .avi files to mp4 format.
    
    Parameters
    ----------
    avi_files : list
        List of paths to .avi files to convert
    """
    if not avi_files:
        print("No files provided to convert.")
        return
    
    print("\nFiles to be converted:")
    for file in avi_files:
        print(f"- {os.path.basename(file)}")
    
    # Ask for confirmation
    response = input(f"\nConvert {len(avi_files)} files? (y/n): ")
    if response.lower() != 'y':
        print("Conversion cancelled.")
        return
    
    # Process each file
    for i, file_path in enumerate(avi_files, 1):
        print(f"\nConverting file {i} of {len(avi_files)}")
        print(f"File: {os.path.basename(file_path)}")
        try:
            convert_for_dlc(file_path)
        except Exception as e:
            print(f"Error converting {file_path}: {str(e)}")
            continue

In [42]:
dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig'
avi_to_convert = find_not_converted_files(dir)
batch_convert_avis(avi_to_convert)



Files to be converted:
- 20250304151443_390L_Reach2.avi
- 20250305175601_771N_Reach2.avi
- 20250305204051_1FB2_Reach2.avi
- 20250304143931_381B_Reach2.avi
- 20250305131847_381B_Reach2.avi
- 20250305142725_390L_Reach2.avi
- 20250304154750_402R_Reach2.avi
- 20250305135657_402R_Reach2.avi
- 20250304162056_771N_Reach2.avi

Converting file 1 of 9
File: 20250304151443_390L_Reach2.avi
Converting: 20250304151443_390L_Reach2.avi
Original size: 7362.8 MB
Converted size: 2522.5 MB
Compression ratio: 2.9x

Converting file 2 of 9
File: 20250305175601_771N_Reach2.avi
Converting: 20250305175601_771N_Reach2.avi
Original size: 4311.1 MB
Converted size: 1043.4 MB
Compression ratio: 4.1x

Converting file 3 of 9
File: 20250305204051_1FB2_Reach2.avi
Converting: 20250305204051_1FB2_Reach2.avi
Original size: 5269.4 MB
Converted size: 1241.9 MB
Compression ratio: 4.2x

Converting file 4 of 9
File: 20250304143931_381B_Reach2.avi
Converting: 20250304143931_381B_Reach2.avi
Original size: 7491.0 MB
Converted siz

### move mp4 files to synology upload folder! it is set up to be a one way sync. i hope to be able to delete the mp4 files after they are uploaded. 

In [43]:
def get_folder_size(folder_path):
    """
    Calculate the total size of a folder in GB.
    
    Parameters
    ----------
    folder_path : str
        Path to the folder
        
    Returns
    -------
    float
        Size in GB
    """
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024 * 1024)  # Convert to GB

In [54]:
def organize_mp4_files(source_dir,destination):
    mp4_paths = get_mp4_path(source_dir)
    if not mp4_files:
        print("No mp4 files found")
        return 
    
    os.makedirs(destination,exist_ok=True)

    #pattern for matching animal IDs (either 111A, 1112A, or 1AA1)
    pattern = re.compile(r"(?:\d{3}[A-Z]|\d[A-Z]{2}\d)")

    for src_path in mp4_paths:
        try:
            filename = os.path.basename(src_path)
            match = pattern.search(filename)
            if not match:
                print(f"could not find animal ID in filename: {filename}")
                continue
            #create animal directory
            animal_ID = match.group(0)
            animal_dir = os.path.join(destination, animal_ID)
            os.makedirs(animal_dir, exist_ok=True)

            #create new file path
            dst_path = os.path.join(animal_dir, filename)

            #move the file 
            if not os.path.exists(dst_path):
                print(f"Moving {filename} to {animal_dir}")
                os.rename(src_path, dst_path)
            else:
                print(f"File already exists: {filename}")
        
        except Exception as e:
            print(f"error moving {src_path}: {str(e)}")
    return


In [55]:
source_dir = r"C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig"
destination = r"C:\Users\Emma\SynologyDrive\cam_recording\behav_rig"
organize_mp4_files(source_dir,destination)

Moving 20250228140652_381B_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\381B
Moving 20250303162758_381B_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\381B
Moving 20250304143931_381B_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\381B
Moving 20250305131847_381B_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\381B
Moving 20250225203846_390L_Day1_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\390L
Moving 20250227183342_390L_Reach_day3.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\390L
Moving 20250228152119_390L_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\390L
Moving 20250303155324_390L_Reach2_struggle.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\390L
Moving 20250305142725_390L_Reach2.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\390L
Moving 20250225200556_402R_Day1_Reach.mp4 to C:\Users\Emma\SynologyDrive\cam_recording\behav_rig\402

### older date fitering before conversion. may not need to implement at base level.

In [30]:
def get_avi_files_after_date(root_dir, start_date):
    """
    Find all .avi files after specified date based on filename timestamp.
    Skips files that don't match YYYYMMDDHHMMSS_*.avi format.
    """
    # Get all avi files
    avi_files = glob.glob(os.path.join(root_dir, '**', '*.avi'), recursive=True)
    
    # Create dataframe with files and their dates
    valid_files = []
    valid_dates = []
    
    for f in avi_files:
        try:
            date = datetime.strptime(os.path.basename(f)[:14], '%Y%m%d%H%M%S')
            valid_files.append(f)
            valid_dates.append(date)
        except (ValueError, IndexError):
            continue
    
    df = pd.DataFrame({
        'filepath': valid_files,
        'filename': [os.path.basename(f) for f in valid_files],
        'date': valid_dates
    })
    
    # Filter by date
    filtered_df = df[df['date'] > start_date]
    filtered_df = filtered_df.sort_values('date')
    
    print(f"Found {len(avi_files)} total .avi files")
    print(f"Found {len(valid_files)} files with valid date format")
    print(f"Found {len(filtered_df)} files after {start_date.strftime('%Y-%m-%d')}")
    
    return filtered_df['filepath'].tolist()

In [31]:
root_dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master'
start_date = datetime(2025, 2, 1)
avi_files = get_avi_files_after_date(root_dir, start_date)

if avi_files:
    print("\nFiles to be converted:")
    for file in avi_files:
        print(f"- {file}")
    
    # Proceed with conversion if desired
   # batch_convert_avis(avi_files)
else:
    print("No files found after specified date")


Found 52 total .avi files
Found 27 files with valid date format
Found 27 files after 2025-02-01

Files to be converted:
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\20250216160904_4934T_Reach2.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\772N\20250216165753_772N_Reach2.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\771N\20250216172646_771N_Reach2.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\771N\20250218114923_771N_Reach2_something.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\771N\20250219164959_771N_Reach2_good.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\771N\20250225171153_771N_expertMB.avi
- C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master\cam_recording\behav_rig\402R\20250225200556_402R_Day1_Reach.avi
- C:\Users\Emma\

In [33]:
root_dir = r'C:\Users\Emma\MIT Dropbox\Emma Odom\Emma\Reach_Task_Master'
start_date = datetime(2025, 2, 1)
batch_convert_filtered_avis(root_dir, start_date)

Found 52 total .avi files
Found 27 files with valid date format
Found 27 files after 2025-02-01

Files to be converted:
- 20250216160904_4934T_Reach2.avi
- 20250216165753_772N_Reach2.avi
- 20250216172646_771N_Reach2.avi
- 20250218114923_771N_Reach2_something.avi
- 20250219164959_771N_Reach2_good.avi
- 20250225171153_771N_expertMB.avi
- 20250225200556_402R_Day1_Reach.avi
- 20250225203846_390L_Day1_Reach2.avi
- 20250226154532_771N_Reach2.avi
- 20250226161553_381B_Reach2.avi
- 20250226171349_402R_Reach2.avi
- 20250227151541_771N_Reach2.avi
- 20250227172608_402R_Reach_day3_great.avi
- 20250227175929_381B_Reach_day3.avi
- 20250227183342_390L_Reach_day3.avi
- 20250228120247_771N_Reach2_furtherst_distance_so_far.avi
- 20250228140652_381B_Reach2.avi
- 20250228144446_402R_Reach2.avi
- 20250228152119_390L_Reach2.avi
- 20250303150140_771N_Reach2.avi
- 20250303155324_390L_Reach2_struggle.avi
- 20250303162758_381B_Reach2.avi
- 20250303170723_402R_Reach2.avi
- 20250303185557_1FB1_LickReach2_day.avi


In [ ]:
#updated version with ability to change compression/quality settings
def convert_to_h264(input_path, output_path=None, quality='lossless'):
    """
    Convert AVI to H.264 MP4 with different quality presets.
    
    Parameters
    ----------
    input_path : str
        Input video path
    output_path : str, optional
        Output path (default: replace .avi with .mp4)
    quality : str
        Quality preset: 'lossless', 'visually_lossless', or 'high_compression'
    """
    if output_path is None:
        output_path = input_path.rsplit('.', 1)[0] + '.mp4'
    
    # Define quality settings
    settings = {
        'lossless': {
            'preset': 'veryslow',
            'qp': 0,
        },
        'visually_lossless': {
            'preset': 'slower',
            'crf': 18,
        },
        'high_compression': {
            'preset': 'slow',
            'crf': 23,
        }
    }
    
    if quality not in settings:
        raise ValueError(f"Unknown quality preset: {quality}")
    
    # Get settings for chosen quality
    params = settings[quality]
    
    try:
        # Setup ffmpeg stream with chosen settings
        stream = (
            ffmpeg
            .input(input_path)
            .output(output_path,
                   vcodec='libx264',
                   preset=params['preset'],
                   **{k: v for k, v in params.items() if k != 'preset'},
                   an=None)  # no audio
            .overwrite_output()
        )
        
        print(f"Converting with {quality} quality...")
        print(f"Settings: {params}")
        
        # Run conversion
        stream.run(capture_stdout=True, capture_stderr=True)
        
        # Print size comparison
        input_size = os.path.getsize(input_path) / (1024 * 1024)
        output_size = os.path.getsize(output_path) / (1024 * 1024)
        print(f"Original size: {input_size:.1f} MB")
        print(f"Converted size: {output_size:.1f} MB")
        print(f"Compression ratio: {input_size/output_size:.1f}x")
        
    except ffmpeg.Error as e:
        print(f"Error converting {input_path}:")
        print(e.stderr.decode())

# Example usage:
"""
# Lossless (largest files, perfect quality)
convert_to_h264('input.avi', quality='lossless')

# Visually lossless (good compression, virtually indistinguishable)
convert_to_h264('input.avi', quality='visually_lossless')

# High compression (smaller files, still good quality)
convert_to_h264('input.avi', quality='high_compression')
"""